# 02 · Cleaning

**Goal:** turn raw measurements into trustworthy daily series, and be explicit about every decision.

The rules (implemented and unit-tested in `src/clean.py`):

| Problem | What we do | Why |
|---|---|---|
| `stato = NA` (ARPA flags the value as invalid) | set to missing (NaN) | the agency itself says the value is not reliable |
| `valore = -9999` | set to missing | ARPA's code for "no measurement"; left in, one such value would destroy an average |
| negative values | set to missing | concentrations cannot be negative |
| absurdly high values (> 1000 µg/m³) | set to missing | instrument glitches; far above any real Milan value |
| NO2 stuck at the same value ≥ 6 hours | set to missing | a frozen analyser, not real air |
| hourly NO2 timestamps are "hour-ending" | shift back 1 hour | so each value lands on the day it was measured |
| days with < 18 valid hours of NO2 | daily mean = missing | the usual EU 75% completeness rule |

**We never delete rows and never invent values.** Invalid data becomes NaN so gaps stay visible, and each later step decides how to treat them.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src.config import DATA_RAW, DATA_PROCESSED, POLLUTANTS
from src.fetch import load_raw
from src.clean import (parse_raw, apply_validity, flag_stuck_values, hour_ending_to_start,
                       daily_mean_from_hourly, to_daily_wide, city_mean)

# Pitfall: by default pandas' read_csv turns the text "NA" into a missing value. Here "NA"
# is ARPA's flag for "not valid", so load_raw switches that off and we keep the real flag.
raw = load_raw()
df = parse_raw(raw)
df = apply_validity(df)
len(df)

537866

## 1. Validity flags and error codes

In [2]:
reasons = (df.groupby(["pollutant"])["invalid_reason"]
             .value_counts(dropna=False).unstack(fill_value=0))
reasons["share_invalid_%"] = (100 * (1 - reasons.get(np.nan, 0) / reasons.sum(axis=1))).round(2)
reasons

invalid_reason,flag_invalid,NaN,share_invalid_%
pollutant,,,
NO2,18019,498706,3.49
PM10,352,12364,2.77
PM2.5,219,8206,2.60


In [3]:
# Do the flag and the error code agree? (every -9999 should also be flagged NA)
pd.crosstab(raw["stato"], raw["valore"] == -9999, rownames=["stato"], colnames=["valore == -9999"])

valore == -9999,False,True
stato,,
NA,0,18590
VA,519276,0


The flag and the error code agree perfectly: every invalid row has `-9999`, and no valid row does. Overall 2.6% (PM2.5) to 3.5% (NO2) of rows are invalid. Also, **no valid value is negative or implausibly high**, so those rules removed nothing; they are there as safeguards in case future data contains such errors.

## 2. Stuck sensors (hourly NO2 only)

In [4]:
hourly = df[df["pollutant"] == "NO2"].copy()
daily_raw = df[df["pollutant"] != "NO2"].copy()

hourly = flag_stuck_values(hourly)
print("NO2 hours removed as 'stuck':", (hourly["invalid_reason"] == "stuck").sum())

NO2 hours removed as 'stuck': 0


None found: the longest run of identical hourly values in any NO2 sensor is 5 hours, which is normal when the air is stable. The check stays in the pipeline as a safeguard.

## 3. Timestamps

ARPA labels each hourly value by the **end** of the hour: `01:00` means the average over 00:00–01:00, and `00:00` is the last hour of the previous day. We shift NO2 back one hour.

ARPA also uses a **fixed clock with no summer-time change**: on daylight-saving days there are still exactly 24 hourly values. So in summer, an NO2 "8:00" peak is 9:00 on the wall clock. We keep this fixed clock and note it under the daily-cycle chart.

In [5]:
hourly["data"] = hour_ending_to_start(hourly["data"])
hourly = hourly[hourly["data"] >= "2015-01-01"]  # the hour ending 2015-01-01 00:00 belongs to 2014
check = hourly[(hourly["idsensore"] == "5531") & (hourly["data"].dt.strftime("%Y-%m-%d") == "2019-03-31")]
print("hours on the DST day 2019-03-31:", len(check))

hours on the DST day 2019-03-31: 24


## 4. Daily means (with the 75% completeness rule)

In [6]:
no2_daily = daily_mean_from_hourly(hourly)
print("NO2 station-days rejected for < 18 valid hours:", int(((no2_daily["n_hours"] > 0) & (no2_daily["n_hours"] < 18)).sum()))
no2_daily = no2_daily.drop(columns="n_hours")

pm_daily = daily_raw.assign(date=daily_raw["data"].dt.normalize())[["idsensore", "date", "valore"]]
daily = pd.concat([pm_daily, no2_daily], ignore_index=True)
daily = parse_raw(daily.rename(columns={"date": "data"})).rename(columns={"data": "date"})
daily = daily[["date", "pollutant", "station", "idsensore", "valore"]]
daily.head()

NO2 station-days rejected for < 18 valid hours: 397


,date,pollutant,station,idsensore,valore
0,2015-01-01,PM10,Milano Pascal Città Studi,10273,65.0
1,2015-01-02,PM10,Milano Pascal Città Studi,10273,69.0
2,2015-01-03,PM10,Milano Pascal Città Studi,10273,84.0
3,2015-01-04,PM10,Milano Pascal Città Studi,10273,40.0
4,2015-01-05,PM10,Milano Pascal Città Studi,10273,27.0


## 5. A consistency check: PM2.5 should never exceed PM10

PM2.5 (particles < 2.5 µm) is a subset of PM10 (particles < 10 µm), so on the same day at the same station PM2.5 ≤ PM10. Two stations measure both.

In [7]:
pm = daily[daily["pollutant"].isin(["PM10", "PM2.5"])].pivot_table(index=["date", "station"], columns="pollutant", values="valore").dropna()
bad = pm[pm["PM2.5"] > pm["PM10"]]
print(f"{len(bad)} of {len(pm)} station-days ({100*len(bad)/len(pm):.1f}%) have PM2.5 > PM10")
print("median excess:", round((bad["PM2.5"] - bad["PM10"]).median(), 1), "µg/m³")

57 of 8148 station-days (0.7%) have PM2.5 > PM10
median excess: 2.0 µg/m³


About 1% of days break this rule, by a few µg/m³. The two pollutants are measured by different instruments, each with its own uncertainty, so small inversions are expected noise rather than errors. We **keep** these days; removing them would bias PM2.5 down and PM10 up on exactly the days when most particles are fine.

## 6. Completeness by year

In [8]:
daily["year"] = daily["date"].dt.year
days_in_year = daily.groupby("year")["date"].nunique()
coverage = (daily.groupby(["pollutant", "station", "year"])["valore"].count()
              .unstack("year").div(days_in_year, axis=1).mul(100).round(0))
coverage

year                                 2015  2016  2017  2018  2019   2020  \
pollutant station                                                          
NO2       Milano Pascal Città Studi  96.0  87.0  95.0  92.0  88.0   95.0   
          Milano Verziere            96.0  98.0  98.0  87.0  98.0  100.0   
          Milano v.Liguria           99.0  88.0  90.0  92.0  98.0   95.0   
          Milano v.Marche            99.0  98.0  94.0  94.0  95.0   96.0   
          Milano v.Senato            98.0  96.0  97.0  89.0  95.0   98.0   
PM10      Milano Pascal Città Studi  94.0  93.0  90.0  95.0  93.0   96.0   
          Milano Verziere            92.0  96.0  97.0  95.0  92.0   95.0   
          Milano v.Senato            98.0  96.0  98.0  97.0  98.0   98.0   
PM2.5     Milano Pascal Città Studi  94.0  90.0  90.0  93.0  93.0   97.0   
          Milano v.Senato            98.0  94.0  96.0  97.0  98.0   98.0   

year                                 2021  2022   2023  2024   2025   2026  
pollutant station                                                           
NO2       Milano Pascal Città Studi  97.0  90.0   97.0  99.0   96.0   96.0  
          Milano Verziere            93.0  99.0  100.0  99.0   91.0  100.0  
          Milano v.Liguria           98.0  90.0   95.0  90.0  100.0   99.0  
          Milano v.Marche            95.0  98.0   92.0  94.0   93.0  100.0  
          Milano v.Senato            92.0  97.0   97.0  95.0   96.0   95.0  
PM10      Milano Pascal Città Studi  94.0  92.0   98.0  97.0   99.0   96.0  
          Milano Verziere            92.0  98.0   98.0  91.0   95.0   98.0  
          Milano v.Senato            89.0  92.0   95.0  93.0   96.0   94.0  
PM2.5     Milano Pascal Città Studi  95.0  93.0   98.0  96.0   99.0   96.0  
          Milano v.Senato            88.0  91.0   94.0  92.0   96.0   96.0

Every station-year from 2015 to 2025 is at least 87% complete, and most are ≥ 90% (the EU target for an annual mean). The 2026 column is relative to the days observed so far.

## 7. City-level daily series

For each pollutant we average the stations on each day, **requiring at least 2 valid stations** (PM2.5 only has 2, so both are needed). This stops one station's gap from changing what "Milan" means on a given day.

In [9]:
city = pd.DataFrame({p: city_mean(to_daily_wide(daily, p), min_stations=2) for p in POLLUTANTS})
city.index.name = "date"
print(city.isna().mean().mul(100).round(1).rename("% days missing"))
# PM2.5 has only two stations, so a day is missing whenever either is down (~11% of days).
# We accept that rather than let "Milan PM2.5" flip between a traffic and a background site.

PM10      0.7
PM2.5    10.6
NO2       0.0
Name: % days missing, dtype: float64


In [10]:
city.describe().round(1)

,PM10,PM2.5,NO2
count,4309.0,3880.0,4337.0
mean,32.1,23.0,42.0
std,20.4,17.0,18.7
min,4.7,2.0,7.9
25%,18.0,11.5,27.4
50%,26.0,17.0,39.4
75%,40.3,30.0,53.5
max,187.3,147.0,118.6


In [11]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
daily.drop(columns="year").to_parquet(DATA_PROCESSED / "daily_station.parquet")
city.to_parquet(DATA_PROCESSED / "daily_city.parquet")
hourly[["data", "station", "valore"]].rename(columns={"data": "hour_start"}).to_parquet(DATA_PROCESSED / "hourly_no2.parquet")
print("saved:", sorted(p.name for p in DATA_PROCESSED.iterdir()))

saved: ['daily_city.parquet', 'daily_station.parquet', 'hourly_no2.parquet']


## Summary

- Invalid values are clearly marked by ARPA (`stato = NA` ↔ `-9999`); 2.6–3.5% of rows, depending on the pollutant, become missing.
- Pitfall avoided: pandas reads the text "NA" as missing by default; `load_raw` keeps it as the real flag.
- No negative, implausible or stuck values exist in this period; those checks are safeguards.
- NO2 timestamps were moved from hour-ending to hour-starting; ARPA uses a fixed clock with no daylight saving.
- Daily NO2 means require ≥ 18 valid hours; city means require ≥ 2 valid stations.
- Output: `daily_station.parquet` (per station), `daily_city.parquet` (city average), `hourly_no2.parquet` (for the daily cycle).